In [1]:
import pandas as pd
import pingouin as pg
from pathlib import Path

In [2]:
!pip install pingouin


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1.Đọc Dữ Liệu Sạch

In [3]:
df = pd.read_excel(Path("..") / "data" / "processed" / "clean_data.xlsx")

print("Kích thước dữ liệu:", df.shape)

print("\nCác biến:")
print(df.columns.tolist())

Kích thước dữ liệu: (720, 7)

Các biến:
['Fragrance', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Moisture', 'Origin']


## 2.Khai Báo Các Biến MANOVA

In [4]:
dependent_vars = [
    "Fragrance",
    "Flavor",
    "Aftertaste",
    "Acidity",
    "Body",
    "Moisture"
]

group_var = "Origin"

print("\nBiến phụ thuộc:")
print(dependent_vars)

print("\nBiến phân nhóm:")
print(group_var)


Biến phụ thuộc:
['Fragrance', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Moisture']

Biến phân nhóm:
Origin


##3.Kiểm Tra Missing (Lần cuối)

In [5]:
missing = df[
    dependent_vars + [group_var]
].isnull().sum()

print(missing)

if missing.sum() == 0:
    print("\n✓ Không còn dữ liệu missing.")
else:
    print("\n⚠ Vẫn còn dữ liệu missing.")


Fragrance     0
Flavor        0
Aftertaste    0
Acidity       0
Body          0
Moisture      0
Origin        0
dtype: int64

✓ Không còn dữ liệu missing.


## 4.Thống Kê Số Lượng Mẫu Theo Nhóm

In [6]:
print("SỐ LƯỢNG QUAN SÁT THEO ORIGIN")

group_counts = df[group_var].value_counts()

print(group_counts)

SỐ LƯỢNG QUAN SÁT THEO ORIGIN
Origin
Mexico       236
Colombia     180
Guatemala    175
Brazil       129
Name: count, dtype: int64


## 5.HENZE-ZIRKLER

Mục đích:


*   Kiểm tra 6 biến phụ thuộc có tuân theo
*   Phân phối chuẩn đa biến hay không.

H0:
**Dữ liệu có phân phối chuẩn đa biến.**

Nếu p > 0.05:
* Không bác bỏ H0 -> giả định chuẩn đa biến phù hợp.

Nếu p <= 0.05:
* Bác bỏ H0 -> dữ liệu có dấu hiệu không chuẩn đa biến.


### 5.1. HZ trên toàn bộ dữ liệu

In [7]:
Y = df[dependent_vars]

hz_all = pg.multivariate_normality(
    Y,
    alpha=0.05
)

print("\n--- Henze-Zirkler toàn bộ dữ liệu ---")

print("HZ statistic :", hz_all.hz)
print("p-value      :", hz_all.pval)
print("Normal       :", hz_all.normal)


if hz_all.pval > 0.05:
    print("Kết luận: Chưa bác bỏ H0.")
    print("→ Dữ liệu phù hợp với giả định chuẩn đa biến.")
else:
    print("Kết luận: Bác bỏ H0.")
    print("→ Dữ liệu có dấu hiệu không tuân theo chuẩn đa biến.")




--- Henze-Zirkler toàn bộ dữ liệu ---
HZ statistic : 4.79770454135916
p-value      : 0.0
Normal       : False
Kết luận: Bác bỏ H0.
→ Dữ liệu có dấu hiệu không tuân theo chuẩn đa biến.


###5.2. HZ riêng theo từng Origin
Kiểm tra trong từng nhóm vì MANOVA so sánh các nhóm Origin với nhau.

In [8]:
print("\n--- Henze-Zirkler theo từng Origin ---")

hz_results = []

for origin, group_data in df.groupby(group_var):

    data_group = group_data[dependent_vars].copy()

    result = pg.multivariate_normality(
        data_group,
        alpha=0.05
    )

    hz_results.append({
        "Origin": origin,
        "n": len(data_group),
        "HZ": result.hz,
        "p_value": result.pval,
        "Normal": result.normal
    })

    print(f"\nOrigin: {origin}")
    print("Số mẫu    :", len(data_group))
    print("HZ        :", result.hz)
    print("p-value   :", result.pval)
    print("Normal    :", result.normal)

    if result.pval > 0.05:
        print("→ Chưa bác bỏ H0: phù hợp chuẩn đa biến.")
    else:
        print("→ Bác bỏ H0: có dấu hiệu không chuẩn đa biến.")


# Chuyển kết quả thành DataFrame
hz_results_df = pd.DataFrame(hz_results)




--- Henze-Zirkler theo từng Origin ---

Origin: Brazil
Số mẫu    : 129
HZ        : 2.6588068047282225
p-value   : 4.64752667658796e-93
Normal    : False
→ Bác bỏ H0: có dấu hiệu không chuẩn đa biến.

Origin: Colombia
Số mẫu    : 180
HZ        : 1.790845364206402
p-value   : 8.212311299396097e-42
Normal    : False
→ Bác bỏ H0: có dấu hiệu không chuẩn đa biến.

Origin: Guatemala
Số mẫu    : 175
HZ        : 2.6368999816738237
p-value   : 4.684893894398519e-101
Normal    : False
→ Bác bỏ H0: có dấu hiệu không chuẩn đa biến.

Origin: Mexico
Số mẫu    : 236
HZ        : 2.2622659017929765
p-value   : 7.567622095241979e-82
Normal    : False
→ Bác bỏ H0: có dấu hiệu không chuẩn đa biến.


##6. BOX'S M
**Mục đích:**
* Kiểm tra xem ma trận hiệp phương sai của 6 biến phụ thuộc
* có giống nhau giữa 4 nhóm Origin hay không.

H0:
**Các nhóm có ma trận hiệp phương sai bằng nhau.**

 Nếu p > 0.05:
* Không bác bỏ H0 -> giả định đồng nhất covariance phù hợp.

 Nếu p <= 0.05:
* Bác bỏ H0 -> covariance giữa các nhóm khác nhau.

In [9]:
print("KIỂM ĐỊNH BOX'S M")

boxm_result = pg.box_m(
    data=df,
    dvs=dependent_vars,
    group=group_var
)

print("\nKết quả Box's M:")
print(boxm_result)

KIỂM ĐỊNH BOX'S M

Kết quả Box's M:
           Chi2    df          pval  equal_cov
box  380.195198  63.0  7.216014e-47      False


##7. LẤY P-VALUE BOX'S M

In [10]:
# Pingouin trả về DataFrame với cột pval
boxm_pvalue = boxm_result["pval"].iloc[0]

print("\nBox's M p-value:", boxm_pvalue)


if boxm_pvalue > 0.05:
    print("Kết luận: Chưa bác bỏ H0.")
    print("→ Có thể xem giả định đồng nhất ma trận hiệp phương sai là phù hợp.")
else:
    print("Kết luận: Bác bỏ H0.")
    print("→ Có bằng chứng cho thấy ma trận hiệp phương sai khác nhau giữa các nhóm.")



Box's M p-value: 7.216014407656887e-47
Kết luận: Bác bỏ H0.
→ Có bằng chứng cho thấy ma trận hiệp phương sai khác nhau giữa các nhóm.


##8. TỔNG HỢP KẾT QUẢ

In [11]:
print("TỔNG HỢP KIỂM TRA GIẢ ĐỊNH MANOVA")

print("\n1. Henze-Zirkler toàn bộ dữ liệu")
print("   HZ      =", hz_all.hz)
print("   p-value =", hz_all.pval)

if hz_all.pval > 0.05:
    hz_conclusion = "Phù hợp chuẩn đa biến"
else:
    hz_conclusion = "Không phù hợp chuẩn đa biến"

print("   Kết luận:", hz_conclusion)


print("\n2. Box's M")
print("   p-value =", boxm_pvalue)

if boxm_pvalue > 0.05:
    boxm_conclusion = "Phù hợp giả định đồng nhất covariance"
else:
    boxm_conclusion = "Không phù hợp giả định đồng nhất covariance"

print("   Kết luận:", boxm_conclusion)

TỔNG HỢP KIỂM TRA GIẢ ĐỊNH MANOVA

1. Henze-Zirkler toàn bộ dữ liệu
   HZ      = 4.79770454135916
   p-value = 0.0
   Kết luận: Không phù hợp chuẩn đa biến

2. Box's M
   p-value = 7.216014407656887e-47
   Kết luận: Không phù hợp giả định đồng nhất covariance


## 9. LƯU KẾT QUẢ KIỂM ĐỊNH

In [12]:
RESULT_FILE = Path("..") / "data" / "processed" / "assumption_results.xlsx"

with pd.ExcelWriter(RESULT_FILE, engine="openpyxl") as writer:

    # HZ theo nhóm
    hz_results_df.to_excel(
        writer,
        sheet_name="Henze_Zirkler",
        index=False
    )

    # Box's M
    boxm_result.to_excel(
        writer,
        sheet_name="Box_M",
        index=False
    )

print("HOÀN THÀNH KIỂM TRA GIẢ ĐỊNH")

print("File kết quả:")
print(RESULT_FILE)

HOÀN THÀNH KIỂM TRA GIẢ ĐỊNH
File kết quả:
..\data\processed\assumption_results.xlsx
